In [ ]:
import pandas as pd

people_df = pd.read_csv("C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/인구밀도 데이터(2024)/전국 지역별(행정동) 성별 연령별 주민등록 인구수(2024).csv", encoding='cp949')
greencar_df = pd.read_csv("C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/차고지 위치 데이터/그린카 차고지 위치 데이터.csv", encoding='utf-8')
socar_df = pd.read_csv("C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/차고지 위치 데이터/쏘카 차고지 위치 데이터(크롤링).csv", encoding='utf-8')

지역별 차고지 수 / 지역별 인구 수

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import koreanize_matplotlib
import warnings

# 경고 무시 설정
warnings.filterwarnings("ignore")

# 한글 폰트 설정
matplotlib.rcParams['font.family'] = 'Malgun Gothic'  # 윈도우에서는 'Malgun Gothic'을 사용합니다.
# matplotlib.rcParams['font.family'] = 'AppleGothic'  # macOS에서는 'AppleGothic'을 사용합니다.
# matplotlib.rcParams['font.family'] = 'NanumGothic'  # Ubuntu와 같은 리눅스에서는 'NanumGothic'을 사용합니다.
matplotlib.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

In [ ]:
people_df

In [ ]:
greencar_df

In [ ]:
socar_df

In [ ]:
# 컬럼명 통일
people_df = people_df.rename(columns={'시도명': '지역명'})

In [ ]:
people_all = people_df.groupby('지역명')['계'].sum().reset_index()

# 결과 출력
print(people_all)

In [ ]:
greencar_df['지역명'] = greencar_df['Address'].apply(lambda x: x.split()[0])

print(greencar_df)

In [ ]:
greencar_df['지역명'].unique()

In [ ]:
socar_df["지역명"].unique()

In [ ]:
socar_all = socar_df['지역명'].value_counts()
socar_all

In [ ]:
greencar_all = greencar_df['지역명'].value_counts()

# 결과 출력
print(greencar_all)

In [ ]:
# '서울특별시'를 '서울'로 통합합니다.
greencar_df['지역명'] = greencar_df['지역명'].replace('서울특별시', '서울')

# '평양'을 삭제합니다.
greencar_df = greencar_df[greencar_df['지역명'] != '평양']

# 변경된 DataFrame을 확인합니다.
greencar_all = greencar_df['지역명'].value_counts()
print(greencar_all)

In [ ]:
socar_all = socar_all.reset_index()
socar_all.columns = ['지역명', '계']

greencar_all = greencar_all.reset_index()
greencar_all.columns = ['지역명', '계']

In [ ]:
# 지역명 표준화
def standardize_region_name(region_name):
    mapping = {
        '서울특별시': '서울',
        '경기도': '경기',
        '경상남도': '경남',
        '경상북도': '경북',
        '광주광역시': '광주',
        '대구광역시': '대구',
        '대전광역시': '대전',
        '부산광역시': '부산',
        '세종특별자치시': '세종',
        '울산광역시': '울산',
        '인천광역시': '인천',
        '전라남도': '전남',
        '전북특별자치도': '전북',
        '제주특별자치도': '제주',
        '충청남도': '충남',
        '충청북도': '충북',
        "강원특별자치도" : "강원"
    }
    return mapping.get(region_name, region_name)

# 데이터프레임에 지역명 표준화 적용
socar_all['지역명'] = socar_all['지역명'].apply(standardize_region_name)
greencar_all['지역명'] = greencar_all['지역명'].apply(standardize_region_name)
people_all['지역명'] = people_all['지역명'].apply(standardize_region_name)


In [ ]:
socar_all

In [ ]:
greencar_all # 광주합치고, 서울합치고, 평양 제거

In [ ]:
 # 광주 합치고

In [ ]:
# 데이터프레임을 지역명 기준으로 병합합니다.
merged_df = people_all.merge(socar_all, on='지역명', how='outer', suffixes=('', '_쏘카'))
merged_df = merged_df.merge(greencar_all, on='지역명', how='outer', suffixes=('', '_그린카'))

# 컬럼 이름을 변경합니다.
merged_df.rename(columns={
    '계': '인구수 합계',
    '계_쏘카': '쏘카지점 합계',
    '계_그린카': '그린카지점 합계'
}, inplace=True)

# 결과 출력
print(merged_df)

In [ ]:
# 차고지 수를 계산합니다.
merged_df['차고지 수'] = merged_df['쏘카지점 합계'] + merged_df['그린카지점 합계']

# 비율을 계산합니다.
merged_df['비율'] = merged_df['차고지 수'] / merged_df['인구수 합계']

# 결과 출력
print(merged_df[['지역명', '인구수 합계', '차고지 수', '비율']])

In [ ]:
# 차고지 수를 계산합니다.
merged_df['차고지 수'] = merged_df['쏘카지점 합계'] + merged_df['그린카지점 합계']

# 전체 인구수와 전체 차고지 수 계산
total_population = merged_df['인구수 합계'].sum()
total_parking_spots = merged_df['차고지 수'].sum()

# 지역별 인구수 비율과 차고지 수 비율 계산
merged_df['인구수 비율'] = merged_df['인구수 합계'] / total_population
merged_df['차고지 비율'] = merged_df['차고지 수'] / total_parking_spots

merged_df['차고지 비율/인구수 비율'] = merged_df['차고지 비율'] / merged_df['인구수 비율']

merged_df

In [ ]:
# 그래프를 위해 비율을 기준 1과 비교하여 편차를 계산
merged_df['편차'] = merged_df['차고지 비율/인구수 비율'] - 1

# 막대그래프 생성
plt.figure(figsize=(14, 8))
plt.bar(merged_df['지역명'], merged_df['편차'], color='skyblue', edgecolor='black')

# 그래프의 스타일을 조정합니다.
plt.axhline(y=0, color='gray', linestyle='--')  # 기준선
plt.xlabel('지역명')
plt.ylabel('차고지 비율 / 인구수 비율 - 1')
plt.title('인구수 대비 차고지')
plt.xticks(rotation=45)
plt.grid(axis='y')

# 결과 출력
plt.tight_layout()
plt.show()

In [ ]:
# 그래프를 위해 비율을 기준 1과 비교하여 편차를 계산
merged_df['편차'] = merged_df['차고지 비율/인구수 비율'] - 1

# 편차를 백분율로 변환
merged_df['편차 (%)'] = merged_df['편차'] * 100

# 막대그래프 생성
plt.figure(figsize=(14, 8))
bars = plt.bar(merged_df['지역명'], merged_df['편차 (%)'], color='skyblue', edgecolor='black')

# 각 막대 위에 숫자 표시
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,    # x 위치
        height + 1,                            # y 위치 (막대 위쪽)
        f'{height:.2f}%',                      # 표시할 숫자 (백분율)
        ha='center',                          # 수평 정렬
        va='bottom'                           # 수직 정렬
    )

# 그래프의 스타일을 조정합니다.
plt.axhline(y=0, color='gray', linestyle='--')  # 기준선
plt.xlabel('지역명')
plt.ylabel('차고지 비율 / 인구수 비율 - 1 (%)')
plt.title('인구수 대비 차고지 (%)')
plt.xticks(rotation=45)
plt.grid(axis='y')

# 결과 출력
plt.tight_layout()
plt.show()

In [ ]:
# 그래프를 위해 비율을 기준 1과 비교하여 편차를 계산
merged_df['편차'] = merged_df['차고지 비율/인구수 비율'] - 1

# 편차를 백분율로 변환
merged_df['편차 (%)'] = merged_df['편차'] * 100

# 편차 기준으로 정렬
sorted_df = merged_df.sort_values(by='편차 (%)')

# 인구수 대비 차고지가 적은 지역 출력
print(sorted_df[['지역명', '인구수 합계', '차고지 수', '편차 (%)']])

# 그래프를 생성합니다 (선택 사항)
plt.figure(figsize=(14, 8))
bars = plt.bar(sorted_df['지역명'], sorted_df['편차 (%)'], color='skyblue', edgecolor='black')

# 각 막대 위에 숫자 표시
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,    # x 위치
        height + 1,                            # y 위치 (막대 위쪽)
        f'{height:.2f}%',                      # 표시할 숫자 (백분율)
        ha='center',                          # 수평 정렬
        va='bottom'                           # 수직 정렬
    )

# 그래프의 스타일을 조정합니다.
plt.axhline(y=0, color='gray', linestyle='--')  # 기준선
plt.xlabel('지역명')
plt.ylabel('차고지 비율 / 인구수 비율 - 1 (%)')
plt.title('인구수 대비 차고지 비율의 편차 (%)')
plt.xticks(rotation=45)
plt.grid(axis='y')

# 결과 출력
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# 순위별 색상을 생성하는 함수
def colors_from_ranks(values, palette_name):
    # 순위를 계산합니다 (작은 값부터 큰 값까지 0, 1, 2, ...)
    ranks = pd.Series(values).rank(method='min').astype(int) - 1
    # 최대 순위를 기준으로 색상을 생성
    palette = sns.color_palette(palette_name, len(values))
    # 순위를 색상에 매핑
    return np.array(palette).take(ranks, axis=0)

# 편차 계산
merged_df['편차'] = merged_df['차고지 비율/인구수 비율'] - 1

# 편차를 백분율로 변환
merged_df['편차 (%)'] = merged_df['편차'] * 100

# 편차 기준으로 정렬
sorted_df = merged_df.sort_values(by='편차 (%)')

# 평균 계산
mean_value = sorted_df['편차 (%)'].mean()

# 그래프를 생성합니다
plt.figure(figsize=(14, 8))

# 순위에 따라 색상을 지정합니다
colors = colors_from_ranks(sorted_df['편차 (%)'], "YlOrRd")

# 막대 그래프를 생성하며, 색상을 적용합니다
bars = plt.bar(sorted_df['지역명'], sorted_df['편차 (%)'], color=colors, edgecolor='black')

# 각 막대 위에 숫자 표시
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,    # x 위치
        height + 1,                            # y 위치 (막대 위쪽)
        f'{height:.2f}%',                      # 표시할 숫자 (백분율)
        ha='center',                          # 수평 정렬
        va='bottom'                           # 수직 정렬
    )

# 평균선 추가
plt.axhline(y=mean_value, color='red', linestyle='--', linewidth=2, label=f'평균: {mean_value:.2f}%')

# 평균 값을 그래프에 텍스트로 표시
plt.text(
    len(sorted_df) - 1, mean_value + 2,       # x: 그래프의 오른쪽 끝, y: 평균선 위에 위치
    f'평균: {mean_value:.2f}%',               # 표시할 텍스트
    color='red',                              # 텍스트 색상
    ha='center', va='bottom',                 # 정렬 설정
    fontsize=12, fontweight='bold'            # 글자 크기와 굵기 설정
)

# 그래프의 스타일을 조정합니다.
plt.axhline(y=0, color='gray', linestyle='--')  # 기준선
plt.xlabel('지역명')
plt.ylabel('차고지 비율 / 인구수 비율 - 1 (%)')
plt.title('인구수 대비 차고지 비율의 편차 (%)')
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.legend()  # 범례 추가

# 결과 출력
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# 색상 팔레트 설정: 색상 리버스
def colors_from_values(values, base_color):
    # Normalize the values to range [0, 1]
    normalized = (values - min(values)) / (max(values) - min(values))
    # Create a color palette from the base color and reverse it
    palette = sns.light_palette(base_color, n_colors=len(values))[::-1]  # Reverse the palette
    return np.array(palette)

# 편차 계산
merged_df['편차'] = merged_df['차고지 비율/인구수 비율'] - 1

# 편차를 백분율로 변환
merged_df['편차 (%)'] = merged_df['편차'] * 100

# 편차 기준으로 정렬
sorted_df = merged_df.sort_values(by='편차 (%)')

# 평균 계산
mean_value = sorted_df['편차 (%)'].mean()

# 그래프를 생성합니다
plt.figure(figsize=(14, 8))

# 순위에 따라 색상을 지정합니다 (Reversed palette with shades of "#00a4ef")
colors = colors_from_values(sorted_df['편차 (%)'], "#00a4ef")

# 막대 그래프를 생성하며, 색상을 적용합니다
bars = plt.bar(sorted_df['지역명'], sorted_df['편차 (%)'], color=colors)  # edgecolor 옵션 제거

# 각 막대 위에 숫자 표시
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,    # x 위치
        height + 1,                            # y 위치 (막대 위쪽)
        f'{height:.2f}%',                      # 표시할 숫자 (백분율)
        ha='center',                          # 수평 정렬
        va='bottom'                           # 수직 정렬
    )

# 평균선 추가
plt.axhline(y=mean_value, color='darkblue', linestyle='--', linewidth=2, label=f'평균: {mean_value:.2f}%')

# 평균 값을 그래프에 텍스트로 표시
plt.text(
    len(sorted_df) - 1, mean_value + 2,       # x: 그래프의 오른쪽 끝, y: 평균선 위에 위치
    f'평균: {mean_value:.2f}%',               # 표시할 텍스트
    color='darkblue',                              # 텍스트 색상
    ha='center', va='bottom',                 # 정렬 설정
    fontsize=12, fontweight='bold'            # 글자 크기와 굵기 설정
)

# 그래프의 스타일을 조정합니다.
plt.axhline(y=0, color='gray', linestyle='--')  # 기준선
plt.xlabel('지역명')
plt.ylabel('차고지 비율 / 인구수 비율 - 1 (%)')
plt.title('인구수 대비 차고지 비율의 편차 (%)')
plt.xticks(rotation=45)
plt.legend()  # 범례 추가

# 결과 출력
plt.tight_layout()
plt.show()
